# 13. Vision transformer and detection blocks

The tensor sizes are small, but each demonstrated algorithm keeps its original computation path. In particular, CenterNet decoding now uses **local-maximum suppression + per-class top-K + global top-K + gathered offset/size heads**, instead of taking one global maximum.


In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device("cpu")
print("device:", device)


## 1. ViT patch sequence


In [ ]:
image = torch.randn(2, 3, 16, 16, device=device)
patch_size = 4
model_dim = 24

patch_projection = nn.Conv2d(
    3,
    model_dim,
    kernel_size=patch_size,
    stride=patch_size,
).to(device)

patch_tokens = patch_projection(image).flatten(2).transpose(1, 2)
cls_token = nn.Parameter(torch.zeros(1, 1, model_dim, device=device))
position = nn.Parameter(
    torch.randn(1, patch_tokens.size(1) + 1, model_dim, device=device) * 0.02
)

vit_tokens = torch.cat(
    [cls_token.expand(image.size(0), -1, -1), patch_tokens],
    dim=1,
)
vit_tokens = vit_tokens + position
print("ViT tokens:", vit_tokens.shape)


## 2. Swin W-MSA / SW-MSA helpers


In [ ]:
def window_partition(x, window_size):
    batch, height, width, channels = x.shape
    x = x.view(
        batch,
        height // window_size,
        window_size,
        width // window_size,
        window_size,
        channels,
    )
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous()
    return x.view(-1, window_size * window_size, channels)


def window_reverse(windows, window_size, height, width, batch):
    channels = windows.size(-1)
    x = windows.view(
        batch,
        height // window_size,
        width // window_size,
        window_size,
        window_size,
        channels,
    )
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous()
    return x.view(batch, height, width, channels)


def relative_position_index(window_size, device):
    coordinates = torch.stack(
        torch.meshgrid(
            torch.arange(window_size, device=device),
            torch.arange(window_size, device=device),
            indexing="ij",
        )
    ).flatten(1)

    relative = coordinates[:, :, None] - coordinates[:, None, :]
    relative = relative.permute(1, 2, 0).contiguous()
    relative[:, :, 0] += window_size - 1
    relative[:, :, 1] += window_size - 1
    relative[:, :, 0] *= 2 * window_size - 1
    return relative.sum(dim=-1)


def build_shifted_window_mask(
    height,
    width,
    window_size,
    shift_size,
    device,
):
    region = torch.zeros(1, height, width, 1, device=device)

    h_slices = (
        slice(0, -window_size),
        slice(-window_size, -shift_size),
        slice(-shift_size, None),
    )
    w_slices = (
        slice(0, -window_size),
        slice(-window_size, -shift_size),
        slice(-shift_size, None),
    )

    region_id = 0
    for h_slice in h_slices:
        for w_slice in w_slices:
            region[:, h_slice, w_slice, :] = region_id
            region_id += 1

    windows = window_partition(region, window_size).squeeze(-1)
    difference = windows[:, None, :] - windows[:, :, None]
    return difference == 0


In [ ]:
class SwinWindowAttention(nn.Module):
    def __init__(
        self,
        model_dim=24,
        heads=3,
        window_size=2,
        shift_size=0,
    ):
        super().__init__()
        self.heads = heads
        self.head_dim = model_dim // heads
        self.window_size = window_size
        self.shift_size = shift_size

        self.norm = nn.LayerNorm(model_dim)
        self.qkv = nn.Linear(model_dim, 3 * model_dim)
        self.out = nn.Linear(model_dim, model_dim)
        self.relative_bias = nn.Parameter(
            torch.zeros((2 * window_size - 1) ** 2, heads)
        )

    def forward(self, x):
        batch, height, width, channels = x.shape
        residual = x
        x = self.norm(x)

        if self.shift_size > 0:
            x = torch.roll(
                x,
                shifts=(-self.shift_size, -self.shift_size),
                dims=(1, 2),
            )

        windows = window_partition(x, self.window_size)
        tokens_per_window = windows.size(1)

        qkv = self.qkv(windows).view(
            windows.size(0),
            tokens_per_window,
            3,
            self.heads,
            self.head_dim,
        ).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)

        scores = q @ k.transpose(-2, -1)
        scores = scores / math.sqrt(self.head_dim)

        index = relative_position_index(self.window_size, x.device)
        bias = self.relative_bias[index.reshape(-1)]
        bias = bias.view(
            tokens_per_window,
            tokens_per_window,
            self.heads,
        ).permute(2, 0, 1)
        scores = scores + bias[None]

        if self.shift_size > 0:
            mask = build_shifted_window_mask(
                height,
                width,
                self.window_size,
                self.shift_size,
                x.device,
            ).repeat(batch, 1, 1)
            scores = scores.masked_fill(
                ~mask[:, None],
                torch.finfo(scores.dtype).min,
            )

        attended = scores.softmax(dim=-1) @ v
        attended = attended.transpose(1, 2).contiguous().flatten(2)
        attended = self.out(attended)

        x = window_reverse(
            attended,
            self.window_size,
            height,
            width,
            batch,
        )

        if self.shift_size > 0:
            x = torch.roll(
                x,
                shifts=(self.shift_size, self.shift_size),
                dims=(1, 2),
            )

        return residual + x


feature = torch.randn(2, 4, 4, 24, device=device)
feature = SwinWindowAttention(shift_size=0).to(device)(feature)
feature = SwinWindowAttention(shift_size=1).to(device)(feature)
print("Swin output:", feature.shape)


## 3. FPN top-down path


In [ ]:
class TinyFPN(nn.Module):
    def __init__(self, channels=(32, 64, 128), out_channels=24):
        super().__init__()
        self.lateral = nn.ModuleList(
            [nn.Conv2d(c, out_channels, 1) for c in channels]
        )
        self.smooth = nn.ModuleList(
            [
                nn.Conv2d(out_channels, out_channels, 3, padding=1)
                for _ in channels
            ]
        )

    def forward(self, features):
        c3, c4, c5 = features
        p5_inner = self.lateral[2](c5)
        p4_inner = self.lateral[1](c4) + F.interpolate(
            p5_inner,
            size=c4.shape[-2:],
            mode="nearest",
        )
        p3_inner = self.lateral[0](c3) + F.interpolate(
            p4_inner,
            size=c3.shape[-2:],
            mode="nearest",
        )

        return (
            self.smooth[0](p3_inner),
            self.smooth[1](p4_inner),
            self.smooth[2](p5_inner),
        )


## 4. CenterNet heads and paper-style decode

CenterNet first suppresses non-local peaks with a 3x3 max-pool, selects top-K candidates, gathers regression heads at those candidate indices, then converts center + offset + size to boxes.


In [ ]:
class CenterNetHead(nn.Module):
    def __init__(self, channels=24, classes=3):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.ReLU(),
        )
        self.heatmap = nn.Conv2d(channels, classes, 1)
        self.offset = nn.Conv2d(channels, 2, 1)
        self.size = nn.Conv2d(channels, 2, 1)

    def forward(self, x):
        hidden = self.shared(x)
        return {
            "heatmap_logits": self.heatmap(hidden),
            "offset": self.offset(hidden),
            "size": F.softplus(self.size(hidden)),
        }


def local_peak_nms(heatmap, kernel=3):
    padding = (kernel - 1) // 2
    local_max = F.max_pool2d(
        heatmap,
        kernel_size=kernel,
        stride=1,
        padding=padding,
    )
    keep = local_max == heatmap
    return heatmap * keep


def topk_centers(heatmap, k):
    batch, classes, height, width = heatmap.shape
    per_class_k = min(k, height * width)

    class_scores, class_indices = torch.topk(
        heatmap.view(batch, classes, -1),
        per_class_k,
        dim=-1,
    )
    class_indices = class_indices % (height * width)
    class_ys = torch.div(class_indices, width, rounding_mode="floor")
    class_xs = class_indices % width

    final_k = min(k, classes * per_class_k)
    scores, flattened_ids = torch.topk(
        class_scores.reshape(batch, -1),
        final_k,
        dim=-1,
    )
    class_ids = torch.div(
        flattened_ids,
        per_class_k,
        rounding_mode="floor",
    )

    flat_indices = class_indices.reshape(batch, -1).gather(1, flattened_ids)
    ys = class_ys.reshape(batch, -1).gather(1, flattened_ids)
    xs = class_xs.reshape(batch, -1).gather(1, flattened_ids)
    return scores, flat_indices, class_ids, ys, xs


def gather_spatial(feature_map, flat_indices):
    batch, channels, height, width = feature_map.shape
    flattened = feature_map.view(batch, channels, height * width)
    gather_index = flat_indices[:, None].expand(-1, channels, -1)
    return flattened.gather(2, gather_index).transpose(1, 2)


def decode_centernet(prediction, k=5):
    heatmap = torch.sigmoid(prediction["heatmap_logits"])
    heatmap = local_peak_nms(heatmap)

    scores, indices, classes, ys, xs = topk_centers(heatmap, k)
    offsets = gather_spatial(prediction["offset"], indices)
    sizes = gather_spatial(prediction["size"], indices)

    center_x = xs.float() + offsets[..., 0]
    center_y = ys.float() + offsets[..., 1]

    x1 = center_x - 0.5 * sizes[..., 0]
    y1 = center_y - 0.5 * sizes[..., 1]
    x2 = center_x + 0.5 * sizes[..., 0]
    y2 = center_y + 0.5 * sizes[..., 1]
    boxes = torch.stack([x1, y1, x2, y2], dim=-1)

    return {
        "scores": scores,
        "classes": classes,
        "boxes": boxes,
    }


## 5. Five-step CPU training check

The feature tensor is fixed so the check isolates the CenterNet prediction heads. The loss uses heatmap logits plus offset and size regression targets; decoding is run after training.


In [ ]:
features = torch.randn(2, 24, 8, 8, device=device)
head = CenterNetHead().to(device)
optimizer = torch.optim.AdamW(head.parameters(), lr=3e-3)

target_heatmap = torch.zeros(2, 3, 8, 8, device=device)
target_heatmap[0, 1, 3, 4] = 1.0
target_heatmap[1, 2, 5, 2] = 1.0

target_offset = torch.zeros(2, 2, 8, 8, device=device)
target_size = torch.ones(2, 2, 8, 8, device=device) * 2.0

loss_history = []
for step in range(5):
    optimizer.zero_grad()
    prediction = head(features)

    heatmap_loss = F.binary_cross_entropy_with_logits(
        prediction["heatmap_logits"],
        target_heatmap,
    )
    offset_loss = F.mse_loss(prediction["offset"], target_offset)
    size_loss = F.mse_loss(prediction["size"], target_size)
    loss = heatmap_loss + 0.1 * offset_loss + 0.1 * size_loss

    loss.backward()
    optimizer.step()

    loss_history.append(loss.item())
    print(f"step {step + 1}: loss={loss.item():.6f}")

detections = decode_centernet(head(features), k=5)
print("loss history:", loss_history)
print("decoded boxes:", detections["boxes"].shape)


## References and provenance

- Swin Transformer: window attention, relative-position bias, cyclic shift, and shifted-window mask.
- FPN: lateral projection, top-down fusion, and smoothing.
- CenterNet / Objects as Points: local-peak NMS, top-K center extraction, gathered regression heads, and center-size box decoding.
